# Exploring the Ribble NFM Candidate Database

Five questions answered directly in SQL against `data/processed/ribble_nfm.duckdb`
— the tabular companion to the spatial GeoPackage. Each question is answered,
then briefly interpreted: what the result actually tells us about the
screening, not just what the query returns.

Rebuild the database with `python3 scripts/build_query_database.py` before
running this notebook if the underlying analysis has changed.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_colwidth", 60)
con = duckdb.connect("../data/processed/ribble_nfm.duckdb", read_only=True)
con.execute("SHOW TABLES").df()

,name
0,candidate_summary
1,final_five
2,flagged_constraints
3,high_confidence_candidates
4,opportunity_gaps
5,provenance
6,recorded_flood_events
7,restoration_projects
8,strong_candidates


## Question 1 — Which candidates are strong, high-confidence, *and* clean?

This is the brief's own example query. It's the most optimistic possible
filter: good score, reliable data, minimal settlement overlap.

In [2]:
con.execute('''
    SELECT candidate_id, nearest_place, composite_score
    FROM candidate_summary
    WHERE band = 'Strong'
      AND data_confidence = 'High'
      AND overlap_built_up_pct < 10
    ORDER BY composite_score DESC
''').df()

,candidate_id,nearest_place,composite_score
0,C16,Sabden,0.570618
1,C19,Sabden,0.570548
2,C20,Barrowford,0.530352
3,C21,Barrowford,0.524281
4,C14,Withnell,0.521286
5,C22,Gisburn,0.491226


**Reading this**: six candidates clear every bar at once — but two of
them (C16, C19) are effectively the same location (both nearest to Sabden,
part of the same tight Pendle/Ribble Valley cluster), and this query doesn't
know that; it can't see geography, only attributes. That's exactly why the
GeoPackage still matters — this table tells you *what* qualifies, not
*where* it sits relative to everything else. SQL and a map answer different
halves of the same question.

## Question 2 — Is the "opportunity gap" story concentrated in weak candidates, or does it hold up in strong ones too?

If distance from recorded restoration activity only shows up in Weak-band
candidates, the "genuine gap" narrative is weaker — it'd just mean the worst
candidates are also the most neglected, which isn't interesting. If it holds
across bands, that's a real finding.

In [3]:
con.execute('''
    SELECT band,
           COUNT(*) AS n_candidates,
           ROUND(AVG(recorded_activity_dist_km), 2) AS avg_dist_to_recorded_activity_km,
           SUM(CASE WHEN recorded_activity_dist_km >= 2.0 THEN 1 ELSE 0 END) AS n_opportunity_gaps
    FROM candidate_summary
    GROUP BY band
    ORDER BY CASE band WHEN 'Strong' THEN 1 WHEN 'Moderate' THEN 2 WHEN 'Watch' THEN 3 ELSE 4 END
''').df()

,band,n_candidates,avg_dist_to_recorded_activity_km,n_opportunity_gaps
0,Strong,9,4.36,6.0
1,Moderate,9,2.90,7.0
2,Watch,9,1.88,4.0
3,Weak,8,2.65,6.0


**Reading this**: opportunity gaps (≥2km from any recorded restoration
project) show up across every band, not just the weak ones — including the
Strong band, where you'd most expect the EA's own project register to have
already caught up with reality if these were well-known sites. That supports
the project's core premise: the modelled potential and the recorded activity
are genuinely not the same map.

## Question 3 — How much of the modelled opportunity is actually unusable as a single "site"?

The Week 3 correction introduced a 1,000ha threshold: anything at or above
it is an "Investigation Zone," not a site. How much of the total modelled
area does that reclassify?

In [4]:
con.execute('''
    SELECT site_type,
           COUNT(*) AS n,
           ROUND(SUM(area_ha)) AS total_area_ha,
           ROUND(100.0 * SUM(area_ha) / SUM(SUM(area_ha)) OVER (), 1) AS pct_of_total_area
    FROM candidate_summary
    GROUP BY site_type
''').df()

,site_type,n,total_area_ha,pct_of_total_area
0,Investigation Zone,5,22593.0,80.7
1,Candidate Site,30,5421.0,19.3


**Reading this**: Investigation Zones are a small minority of
*candidates* (5 of 35) but a large majority of the modelled *area* — most of
the catchment's raw modelled NFM potential, by hectare, sits in oversized
polygons that can't be presented as a single actionable site. That's a
genuinely important caveat for anyone skimming just the headline "35
candidates identified" figure.

## Question 4 — Which candidates look good on paper but need real scrutiny?

The whole reason the protected-site correction mattered: a candidate can be
top-ranked and rank-stable *and* sit on legally protected land at the same
time. This query recreates that finding directly in SQL — candidates in the
top half of the field, fully rank-stable, that still carry a real
constraint flag.

In [5]:
con.execute('''
    SELECT candidate_id, nearest_place, rank, top10_stability, site_type,
           protected_site_flag, overlap_built_up_pct
    FROM candidate_summary
    WHERE rank <= 18
      AND top10_stability >= 3
      AND (
          (protected_site_flag IS NOT NULL AND protected_site_flag != 'None')
          OR overlap_built_up_pct > 8
      )
    ORDER BY rank
''').df()

,candidate_id,nearest_place,rank,top10_stability,site_type,protected_site_flag,overlap_built_up_pct
0,C19,Sabden,2,4,Candidate Site,None,9.561657
1,C17,Chatburn,3,4,Investigation Zone,Clitheroe Knoll Reefs SSSI; Coplow Quarry SSSI; Little M...,10.299021
2,C07,Freckleton,4,3,Candidate Site,None,17.598913
3,C20,Barrowford,5,3,Investigation Zone,Higherford Old Bridge,8.137803
4,C15,Blackburn (Blackburn with Darwen),10,3,Candidate Site,None,59.975782


**Reading this**: C17 sits right at the top of this list — rank 3,
fully rank-stable, and flagged for both a protected-site overlap and
settlement overlap. This is precisely the candidate the Week 3 correction
removed from the final five. A good composite score alone would never have
caught this; it took checking constraint columns explicitly, which is what
this query does directly.

## Question 5 — What do the reference tables say about this catchment's actual flood and restoration history?

`recorded_flood_events` and `restoration_projects` are the non-spatial
attribute tables behind two of the layers used in scoring. On their own,
independent of any candidate, what do they say about the catchment?

In [6]:
flood_causes = con.execute('''
    SELECT flood_caus, COUNT(*) AS n
    FROM recorded_flood_events
    WHERE flood_caus IS NOT NULL AND flood_caus != ''
    GROUP BY flood_caus
    ORDER BY n DESC
    LIMIT 5
''').df()
flood_causes

,flood_caus,n
0,unknown,114
1,channel capacity exceeded (no raised defences),81
2,local drainage/surface water,44
3,other,15
4,overtopping of defences,8


In [7]:
restoration_actions = con.execute('''
    SELECT action, COUNT(*) AS n, ROUND(SUM(amount), 1) AS total_amount
    FROM restoration_projects
    GROUP BY action
    ORDER BY n DESC
''').df()
restoration_actions

,action,n,total_amount
0,Create new habitat,49,160.6
1,Restore habitat features,31,229.3
2,Maintain and improve condition,5,6.3


**Reading this**: the dominant recorded flood cause and the dominant
restoration action type give a sense of what's already understood about this
catchment before any of the candidate screening happens — useful context for
a practitioner conversation, and a sanity check that the reference data
itself is sensible (not empty, not dominated by one bogus category).

## Summary

| Question | One-line takeaway |
|---|---|
| 1 | The brief's own filter query works, but attribute filters alone can't see that two "different" results are geographically redundant |
| 2 | The opportunity-gap pattern holds across bands, not just weak candidates - supports the project's core premise |
| 3 | Most of the modelled *area* is in oversized Investigation Zones, even though most *candidates* are normal-sized sites |
| 4 | SQL alone reconstructs the exact finding that drove the Week 3 correction (C17) - a stable rank isn't the same as a checked one |
| 5 | The reference tables hold up as sensible, independently browsable context, not just scoring inputs |

Close the connection when done:

In [8]:
con.close()